# DevBugger Data Audit

This notebook checks the 800-report dataset before feature selection and modeling. Run the cells in order using the project's Python environment. The CSV can be in the current directory or in the project root one level above it.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASET_NAME = "devbugger_dataset_800.csv"
dataset_path = Path(DATASET_NAME)
if not dataset_path.is_file():
    dataset_path = Path("..") / DATASET_NAME

reports = pd.read_csv(dataset_path)
print(f"{len(reports):,} reports and {reports.shape[1]} columns")

800 reports and 37 columns


## Inspect one report from each source

A pandas DataFrame is a table. Here, each row represents one report. `Bug_Type` is the classification target. `resolution_hours` is the regression target. GitHub durations end at issue closure. Stack Overflow durations end when the accepted answer was posted.

In [2]:
TIME_COLUMNS = ["reported_at_utc", "resolved_at_utc"]
TARGET_COLUMNS = ["Bug_Type", "resolution_hours"]
REPORT_COLUMNS = ["source", "report_title", *TARGET_COLUMNS, *TIME_COLUMNS, "url"]

sample_reports = reports.groupby("source", sort=False).head(1)
display(sample_reports[REPORT_COLUMNS])

,source,report_title,Bug_Type,resolution_hours,reported_at_utc,resolved_at_utc,url
0,GitHub,passport.authenticate doesnt work,Authentication,0.9189,2017-06-08T20:29:49+00:00,2017-06-08T21:24:57+00:00,https://github.com/mikenicholson/passport-jwt/...
550,Stack Overflow,Access-Control-Allow-Origin' header in the res...,CORS_Error,0.7978,2018-02-21T07:01:28+00:00,2018-02-21T07:49:20+00:00,https://stackoverflow.com/questions/48899893/a...


## Recalculate the stored durations

Convert the two timestamp columns to UTC datetimes, subtract them, and divide elapsed seconds by 3,600. The stored target is rounded to four decimal places. We compare the recalculated values at that precision after checking for invalid timestamps.

In [3]:
timestamps = reports[TIME_COLUMNS].apply(
    pd.to_datetime, format="ISO8601", utc=True, errors="coerce"
)
elapsed_hours = (
    timestamps["resolved_at_utc"] - timestamps["reported_at_utc"]
).dt.total_seconds() / 3600

sample_durations = sample_reports[["source", "resolution_hours"]].copy()
sample_durations["calculated_hours"] = elapsed_hours.loc[sample_reports.index].round(4)
display(sample_durations)

,source,resolution_hours,calculated_hours
0,GitHub,0.9189,0.9189
550,Stack Overflow,0.7978,0.7978


## Check dataset integrity and coverage

A repeated URL can indicate a duplicate report. A missing target makes that row unusable for the corresponding supervised learning task. Class and source counts show how the collected dataset is distributed.

In [4]:
comparable = elapsed_hours.notna() & reports["resolution_hours"].notna()
duration_differences = elapsed_hours[comparable] - reports.loc[
    comparable, "resolution_hours"
]

integrity_checks = pd.Series(
    {
        "Missing bug types": reports["Bug_Type"].isna().sum(),
        "Missing resolution times": reports["resolution_hours"].isna().sum(),
        "Missing URLs": reports["url"].isna().sum(),
        "Repeated URLs": reports["url"].duplicated().sum(),
        "Missing or invalid timestamp values": timestamps.isna().sum().sum(),
        "Negative elapsed times": elapsed_hours.lt(0).sum(),
        "Stored durations outside rounding tolerance": (
            duration_differences.abs().gt(0.00005 + 1e-9).sum()
        ),
    },
    name="Count",
)
display(integrity_checks)
display(reports["Bug_Type"].value_counts().rename("Reports by bug type"))
display(reports["source"].value_counts().rename("Reports by source"))

github_reports = reports.loc[reports["source"].eq("GitHub")]
print(f"Distinct GitHub repositories: {github_reports['repository'].nunique()}")

Missing bug types                              0
Missing resolution times                       0
Missing URLs                                   0
Repeated URLs                                  0
Missing or invalid timestamp values            0
Negative elapsed times                         0
Stored durations outside rounding tolerance    0
Name: Count, dtype: int64

Bug_Type
CORS_Error             167
Authentication         162
API_Error              160
State_Management       156
Database_Connection    155
Name: Reports by bug type, dtype: int64

source
GitHub            550
Stack Overflow    250
Name: Reports by source, dtype: int64

Distinct GitHub repositories: 58


## Examine the spread of resolution times

Compare the mean with the median (the 50th percentile). Then compare medians by source and inspect the longest case. A platform timestamp can include time spent waiting for a response or closure.

In [5]:
duration_stats = reports["resolution_hours"].describe()
duration_by_source = (
    reports.groupby("source")["resolution_hours"]
    .agg(["count", "median", "max"])
    .rename(columns={"median": "median_hours", "max": "max_hours"})
)

display(duration_stats.round(2))
display(duration_by_source.round(2))
display(reports.nlargest(1, "resolution_hours")[REPORT_COLUMNS])

count      800.00
mean      3287.32
std       8811.56
min          0.02
25%          1.57
50%         35.46
75%       1601.47
max      83830.93
Name: resolution_hours, dtype: float64

,count,median_hours,max_hours
source,,,
GitHub,550,175.68,83830.93
Stack Overflow,250,1.11,13715.40


,source,report_title,Bug_Type,resolution_hours,reported_at_utc,resolved_at_utc,url
179,GitHub,Unable to recover from mongo connection lost,Database_Connection,83830.9281,2016-07-01T15:04:39+00:00,2026-01-23T14:00:20+00:00,https://github.com/agenda/agenda/issues/338


## Findings

- The dataset has 800 reports, five bug classes, no missing targets, and no repeated URLs. Class counts range from 155 to 167.
- The sample checks reproduced 0.9189 hours for the GitHub authentication report and 0.7978 hours for the Stack Overflow CORS report. These two checks are spot checks of the targets.
- The overall median is 35.46 hours, and the mean is 3,287.32 hours. A long right tail pulls the mean up. The longest report ([Agenda issue #338](https://github.com/agenda/agenda/issues/338)) was flagged for review because the closure date alone does not establish when a fix was made.
- GitHub accounts for 550 reports from 58 repositories. The proposal planned for 20 GitHub repositories. Stack Overflow accounts for 250 reports. The source medians are 175.68 hours and 1.11 hours, respectively. Interpret this gap in light of the different report populations and duration endpoints.
- Use information known at report time as model inputs. `resolved_at_utc` and `log_resolution_hours` encode information about the outcome and would leak the answer into the regression model.